# UDSC 2026 — Task 2 P14 (Kaggle T4 x2)

Attach a Kaggle Dataset containing `task2_p14_code_bundle.zip` and `task2_p14_beam_input.tar.zst`, select **GPU T4 x2**, enable Internet, then choose **Run All**. The historical `beam_input` filename contains provider-neutral data and does not invoke Beam.

In [ ]:
import os
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch

gpu_count = torch.cuda.device_count()
gpu_names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
gpu_capabilities = [torch.cuda.get_device_capability(i) for i in range(gpu_count)]
print('Torch:', torch.__version__)
print('GPU count:', gpu_count)
print('GPU names:', gpu_names)
print('Capabilities:', gpu_capabilities)
assert gpu_count == 2, 'Stop: select GPU T4 x2 and restart the Kaggle session'
assert all(capability >= (7, 0) for capability in gpu_capabilities), 'Stop: P100/sm_60 is incompatible; use T4 x2'
print('T4_X2_PREFLIGHT_PASS')


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

input_root = Path('/kaggle/input')
working = Path('/kaggle/working/udsc2026')
output = Path('/kaggle/working/task2_p14')
working.mkdir(parents=True, exist_ok=True)

code_bundles = list(input_root.rglob('task2_p14_code_bundle.zip'))
code_dirs = [p for p in input_root.rglob('task2_p14_code_bundle') if p.is_dir()]
archives = list(input_root.rglob('task2_p14_beam_input.tar.zst'))
assert len(code_bundles) + len(code_dirs) == 1, f'Expected one code input, found: {code_bundles + code_dirs}'
assert len(archives) == 1, f'Expected one input archive, found: {archives}'

if code_bundles:
    with zipfile.ZipFile(code_bundles[0]) as bundle:
        bundle.extractall(working)
else:
    shutil.copytree(code_dirs[0], working, dirs_exist_ok=True)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'accelerate>=1.1,<2',
        'huggingface-hub>=1.3,<2',
        'nltk>=3.8,<4',
        'peft>=0.17,<1',
        'rouge-score>=0.1,<1',
        'safetensors>=0.5,<1',
        'scikit-learn>=1.5,<2',
        'sentencepiece>=0.2,<1',
        'transformers==5.0.0',
        'zstandard==0.23.0',
    ],
    check=True,
)
# TorchAO is optional for this FP16 LoRA run. Kaggle's preinstalled
# 0.10 release is incompatible with current PEFT, so remove it.
subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'],
    check=False,
)

os.chdir(working)
subprocess.run(
    [
        sys.executable,
        'scripts/cloud/kaggle_task2_p14.py',
        '--archive', str(archives[0]),
        '--workspace', str(working),
        '--output-dir', str(output),
        '--stage', 'all',
        '--epochs', '2',
    ],
    check=True,
)

print('FINAL RESULT:', output / 'task2_p14_kaggle_result.zip')
print('SUBMISSION (only when PUBLIC_CANDIDATE_READY):', output / 'submission.zip')
